##### 재태깅건

- all_pass_ner_tagged_3: 2에서 미태깅된 문장 재태깅한 건으로 deleted_ids 적용해야함.

In [1]:
import json
from collections import defaultdict
from pathlib import Path
import pandas as pd

data_path = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_3")

result = defaultdict(
    lambda: {
        "문서ID": set(),
        "문장ID": set(),
    }
)

for file_path in sorted(data_path.glob("*.json")):
    with file_path.open("r", encoding="utf-8-sig") as file:
        source = json.load(file)

    press = file_path.stem

    for document in source["document"]:
        result[press]["문서ID"].add(document["id"])

        for paragraph in document["paragraph"]:
            for sentence in paragraph["sentence"]:
                result[press]["문장ID"].add(sentence["id"])

table = pd.DataFrame(
    [
        {
            "언론사": press,
            "기사수": len(values["문서ID"]),
            "문장수": len(values["문장ID"]),
        }
        for press, values in result.items()
    ]
)

table = table.reset_index(drop=True)

In [43]:
display(table[["기사수", "문장수"]].sum().to_frame().T)
display(table)

,기사수,문장수
0,121323,2132327


,언론사,기사수,문장수
0,강원일보,1300,18364
1,경기일보,1850,26511
2,경향신문,5597,114296
3,노컷뉴스,7230,121400
4,뉴스1,18869,320285
5,뉴스핌,8648,157918
6,뉴시스,12633,201292
7,디지털타임스,4571,81854
8,마이데일리,4856,95439
9,부산일보,3537,63774


- 삭제 대상

In [50]:
# 삭제 대상 ID 불러오기
df1 = pd.read_csv(r"C:\work\ner\data\org\all_pass_ner_tagged_3\deleted_ids\R5_3737건_exclusion_deleted_ids.csv")
df2 = pd.read_csv(r"C:\work\ner\data\org\all_pass_ner_tagged_3\deleted_ids\서울신문_1143건_exclusion_deleted_ids.csv") 

# 해당 ID에 해당하는 행 삭제
delete_ids_df1 = df1["doc_id"]
delete_ids_df2 = df2["doc_id"]

In [51]:
delete_ids = set(delete_ids_df1.dropna().astype(str)) | set(delete_ids_df2.dropna().astype(str))
deleted_ids_path = data_path / "deleted_ids"
deleted_ids_path.mkdir(exist_ok=True)

verification = []

for file_path in sorted(data_path.glob("*.json")):
    with file_path.open("r", encoding="utf-8-sig") as file:
        source = json.load(file)

    press = file_path.stem
    documents = source["document"]
    remaining_documents = [
        document
        for document in documents
        if str(document["id"]) not in delete_ids
    ]
    deleted_documents = [
        document
        for document in documents
        if str(document["id"]) in delete_ids
    ]

    def count_sentences(document_list):
        return sum(
            len(paragraph["sentence"])
            for document in document_list
            for paragraph in document["paragraph"]
        )

    source["document"] = remaining_documents
    output_path = deleted_ids_path / file_path.name
    output_path.write_text(
        json.dumps(source, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    source_articles = len(documents)
    deleted_articles = len(deleted_documents)
    remaining_articles = len(remaining_documents)
    source_sentences = count_sentences(documents)
    deleted_sentences = count_sentences(deleted_documents)
    remaining_sentences = count_sentences(remaining_documents)

    verification.append({
        "언론사": press,
        "기존 기사수": source_articles,
        "삭제 기사수": deleted_articles,
        "삭제 후 기사수": remaining_articles,
        "기존 문장수": source_sentences,
        "삭제 문장수": deleted_sentences,
        "삭제 후 문장수": remaining_sentences,
        "기사수 검증": source_articles - deleted_articles == remaining_articles,
        "문장수 검증": source_sentences - deleted_sentences == remaining_sentences,
    })

verification_table = pd.DataFrame(verification)
display(verification_table)
display(verification_table[["기존 기사수", "삭제 기사수", "삭제 후 기사수", "기존 문장수", "삭제 문장수", "삭제 후 문장수"]].sum().to_frame().T)

,언론사,기존 기사수,삭제 기사수,삭제 후 기사수,기존 문장수,삭제 문장수,삭제 후 문장수,기사수 검증,문장수 검증
0,강원일보,1300,17,1283,18364,462,17902,True,True
1,경기일보,1850,9,1841,26511,268,26243,True,True
2,경향신문,5597,13,5584,114296,513,113783,True,True
3,노컷뉴스,7230,34,7196,121400,1315,120085,True,True
4,뉴스1,18869,90,18779,320285,2710,317575,True,True
5,뉴스핌,8648,88,8560,157918,3057,154861,True,True
6,뉴시스,12633,49,12584,201292,1392,199900,True,True
7,디지털타임스,4571,10,4561,81854,284,81570,True,True
8,마이데일리,4856,183,4673,95439,9310,86129,True,True
9,부산일보,3537,19,3518,63774,553,63221,True,True


,기존 기사수,삭제 기사수,삭제 후 기사수,기존 문장수,삭제 문장수,삭제 후 문장수
0,121323,1202,120121,2132327,39181,2093146


- all_pass_ner_tagged_2 기준 재태깅 데이터 대체 후 all_pass_ner_tagged_4 저장

In [5]:
# press_list = ["뉴스핌","뉴시스","디지털타임스","마이데일리","부산일보","서울신문","세계일보","스포츠경향","스포츠투데이","아시아투데이","이데일리","이투데이","일간투데이","충청일보"]
press_list = ["충청일보"]

In [6]:
for PRESS in press_list:
    print(PRESS)

    source_path = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_2") / f"{PRESS}.json"
    replacement_path = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_3\deleted_ids") / f"{PRESS}.json"
    output_path = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_4") / f"{PRESS}.json"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with source_path.open("r", encoding="utf-8-sig") as file:
        source = json.load(file)

    with replacement_path.open("r", encoding="utf-8-sig") as file:
        replacement_source = json.load(file)

    source_documents = source["document"]
    replacement_documents = replacement_source["document"]
    replacement_by_id = {str(document["id"]): document for document in replacement_documents}
    source_ids = {str(document["id"]) for document in source_documents}
    replacement_ids = set(replacement_by_id)
    matched_ids = source_ids & replacement_ids
    unmatched_replacement_ids = replacement_ids - source_ids
    matched_source_documents = [
        document for document in source_documents if str(document["id"]) in matched_ids
    ]
    matched_replacement_documents = [replacement_by_id[doc_id] for doc_id in matched_ids]

    def count_sentences(documents):
        return sum(
            len(paragraph["sentence"])
            for document in documents
            for paragraph in document["paragraph"]
        )

    def document_example(document):
        sentences = [
            sentence
            for paragraph in document["paragraph"]
            for sentence in paragraph["sentence"]
        ]
        return {
            "doc_id": document["id"],
            "문장수": len(sentences),
            "첫 문장": sentences[0].get("form", "") if sentences else "",
            "metadata": document.get("metadata", {}),
        }

    updated_documents = [
        replacement_by_id.get(str(document["id"]), document)
        for document in source_documents
    ]
    source["document"] = updated_documents
    output_path.write_text(
        json.dumps(source, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    summary = pd.DataFrame([
        {
            "언론사": PRESS,
            "기존 기사수": len(source_documents),
            "교체 파일 기사수": len(replacement_documents),
            "교체 대상 기사수": len(matched_ids),
            "교체 파일 미매칭 기사수": len(unmatched_replacement_ids),
            "최종 기사수": len(updated_documents),
            "기존 문장수": count_sentences(source_documents),
            # "교체 전 문장수": count_sentences(matched_source_documents),
            # "교체 후 문장수": count_sentences(matched_replacement_documents),
            "최종 문장수": count_sentences(updated_documents),
            "기사수 검증": len(source_documents) == len(updated_documents),
            "문장수 검증": count_sentences(updated_documents) == count_sentences(source_documents) - count_sentences(matched_source_documents) + count_sentences(matched_replacement_documents),
        }
    ])

    display(summary)

    if matched_ids:
        example_id = sorted(matched_ids)[0]
        example = pd.DataFrame(
            [
                {"구분": "기존", **document_example(next(document for document in source_documents if str(document["id"]) == example_id))},
                {"구분": "변경", **document_example(replacement_by_id[example_id])},
            ]
        )
        display(example)

    print(f"저장 완료: {output_path}")

충청일보


,언론사,기존 기사수,교체 파일 기사수,교체 대상 기사수,교체 파일 미매칭 기사수,최종 기사수,기존 문장수,최종 문장수,기사수 검증,문장수 검증
0,충청일보,30173,2032,2032,0,30173,338708,338708,True,True


,구분,doc_id,문장수,첫 문장,metadata
0,기존,NLRW2600000004.10004,20,"지구 온도 1.5도 억제, 식량안보 위협 우려…개발도상국 이중고","{'title': '충청일보 2025년 기사', 'author': '이한영 기자',..."
1,변경,NLRW2600000004.10004,20,"지구 온도 1.5도 억제, 식량안보 위협 우려…개발도상국 이중고","{'title': '충청일보 2025년 기사', 'author': '이한영 기자',..."


저장 완료: C:\work\ner\data\org\all_pass_ner_tagged_4\충청일보.json


---

##### 언론사별 룰 검출

`run_rule_pipeline.ipynb`에서 사용한 룰과 동일한 순서로 실행합니다. 
강원일보 원본 전체를 처리하고, 룰 결과와 `candidate_articles` 확인용 파일을 `data/filter/rule_detect_data/<run_id>`에 저장합니다.
후보 문서는 룰 적중 문장 중 하나라도 true인 문장이 하나 이상 있는 문서입니다. (계엄제외)

In [9]:
import json
from datetime import datetime
from pathlib import Path

import pandas as pd

from src.data_processing.rule_pipeline import load_refer_data
from src.data_processing.prepare.transform import entity_data
from src.data_processing.rules.entity_rules import ext_e001, ext_e002, ext_e003, ext_e004, ext_e005, ext_e006
from src.data_processing.rules.sentence_rules import ext_s001, ext_s002, ext_s003

In [ ]:
DATA_PATH = Path(r"C:\work\ner\data\org\all_pass_ner_tagged_2\노컷뉴스.json")
RULE_OUTPUT_ROOT = Path(r"C:\work\ner\data\filter\rule_detect_data")

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + "_노컷뉴스"
OUTPUT_DIR = RULE_OUTPUT_ROOT / RUN_ID

CHUNK_SIZE = 1000  # 처리 속도에 따라 변경
CHUNK_DIR = RULE_OUTPUT_ROOT / RUN_ID / DATA_PATH.stem # 청크 저장 위치

OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
CHUNK_DIR.mkdir(parents=True, exist_ok=False)

source = json.loads(DATA_PATH.read_text(encoding="utf-8-sig"))
documents = source["document"]
print({"source": len(documents), "to_process": len(documents), "output": str(OUTPUT_DIR)})

{'source': 54008, 'to_process': 54008, 'output': 'C:\\work\\ner\\data\\filter\\rule_detect_data\\20261117_102123_노컷뉴스'}


- 룰 검출

In [31]:
refer_data = load_refer_data()
error_data = refer_data["error_data"]
high_frequency_error_data = error_data.loc[error_data["freq_quartile"].isin(["Q4", "Q3", "Q2", "Q1"])].copy()

from time import perf_counter
from tqdm.auto import tqdm

def sentence_data_from_documents(data, docs):
    source_id = data.get("id", DATA_PATH.stem)
    rows = []
    for doc in tqdm(docs, desc="문서 > 문장 변환"):
        for paragraph in doc["paragraph"]:
            for sentence in paragraph["sentence"]:
                rows.append({
                    "id": source_id,
                    "doc_id": doc["id"],
                    "sentence_id": sentence["id"],
                    "form": sentence["form"],
                })
    return pd.DataFrame(rows)

def run_required_rules(data, docs):
    started = perf_counter()
    output = sentence_data_from_documents(data, docs)
    print(f"문장 변환 완료: {perf_counter() - started:.1f}초")

    sentence_rules = (ext_s001, ext_s002, ext_s003)
    for rule in tqdm(sentence_rules, desc="문장 룰"):
        rule_started = perf_counter()
        output = rule(output)
        print(f"{rule.__name__} 완료: {perf_counter() - rule_started:.1f}초")

    started = perf_counter()
    output = entity_data(output)
    print(f"개체명 변환 완료: {perf_counter() - started:.1f}초")

    entity_rules = (
        (ext_e001, high_frequency_error_data),
        (ext_e002, None),
        (ext_e003, refer_data["mistag_data"]),
        (ext_e004, refer_data["tag_data"]),
        (ext_e005, refer_data["tag_data"]),
        (ext_e006, refer_data["tag_data"]),
    )

    for rule, rule_data in tqdm(entity_rules, desc="개체명 룰"):
        rule_started = perf_counter()
        output = rule(output, rule_data) if rule_data is not None else rule(output)
        print(f"{rule.__name__} 완료: {perf_counter() - rule_started:.1f}초")

    return output

In [32]:
document_chunks = [
    documents[start:start + CHUNK_SIZE]
    for start in range(0, len(documents), CHUNK_SIZE)
]

chunk_paths = []
for chunk_number, document_chunk in enumerate(
    tqdm(document_chunks, desc="문서 청크 처리"),
    start=1,
):
    chunk_result = run_required_rules(source, document_chunk)
    chunk_path = CHUNK_DIR / f"{DATA_PATH.stem}_chunk_{chunk_number:04d}.pkl"
    chunk_result.to_pickle(chunk_path)
    chunk_paths.append(chunk_path)
    print(f"청크 {chunk_number}/{len(document_chunks)} 저장 완료: {chunk_path}")
    del chunk_result

# 청크 Pickle을 병합하여 기존 최종 결과 위치에 저장
rule_result = pd.concat(
    (pd.read_pickle(chunk_path) for chunk_path in chunk_paths),
    ignore_index=True,
)
final_result_path = OUTPUT_DIR / f"{DATA_PATH.stem}.pkl"
rule_result.to_pickle(final_result_path)
print(f"최종 병합 저장 완료: {final_result_path}")
print(rule_result.shape, rule_result.doc_id.nunique())

문서 > 문장 변환: 100%|██████████| 1000/1000 [00:01<00:00, 965.26it/s]


문장 변환 완료: 1.1초


문장 룰: 100%|██████████| 3/3 [00:00<00:00, 89.99it/s]


ext_s001 완료: 0.0초
ext_s002 완료: 0.0초
ext_s003 완료: 0.0초
개체명 변환 완료: 0.3초


ext_e001 완료: 0.0초
ext_e002 완료: 0.0초


ext_e003 완료: 6.9초


... (536 outputs left out)

개체명 룰:  67%|██████▋   | 4/6 [00:00<00:00, 28.94it/s]

ext_e005 완료: 0.0초
ext_e006 완료: 0.0초


문서 청크 처리: 100%|██████████| 55/55 [12:21<00:00, 13.49s/it]


청크 55/55 저장 완료: C:\work\ner\data\filter\rule_detect_data\20261117_102123_노컷뉴스\노컷뉴스\노컷뉴스_chunk_0055.pkl
최종 병합 저장 완료: C:\work\ner\data\filter\rule_detect_data\20261117_102123_노컷뉴스\노컷뉴스.pkl
(1677828, 106) 54008


- 구조화 및 저장
> rule_result가 전체결과, 저장되는것은 1차필터결과

In [10]:
base_columns = ["id", "doc_id", "sentence_id", "entity_id", "form", "entity_tagging", "entity_name", "entity_type"]
rule_columns = [col for col in rule_result.columns if col.startswith(("EXT_S", "EXT_E")) and not col.endswith("_PATTERN")]
rule_pattern_columns = [f"{col}_PATTERN" for col in rule_columns if f"{col}_PATTERN" in rule_result.columns]
save_columns = [
    *[col for col in base_columns if col in rule_result.columns],
    *rule_columns,
    *rule_pattern_columns,
]
rule_result = rule_result.loc[:, save_columns].copy()
rule_result.to_pickle(final_result_path)

rule_hit = rule_result[rule_columns].eq(True).any(axis=1)
martial_law = rule_result["EXT_S001"].eq(True)

# 룰 1개 이상 true & 오류 빈도 높았던 건 (계엄제외)
candidate_sentence = rule_hit & rule_result["EXT_E001"].eq(True) & ~martial_law
candidate_doc_ids = set(rule_result.loc[candidate_sentence, "doc_id"])

# 후보 문장의 룰 결과 확인용 산출물
candidate_columns = [*base_columns, *rule_columns, *rule_pattern_columns]
candidate_rule_result = rule_result.loc[candidate_sentence, [col for col in candidate_columns if col in rule_result.columns]].copy()
candidate_rule_result.to_csv(OUTPUT_DIR / "candidate_rule_results.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"doc_id": sorted(candidate_doc_ids)}).to_csv(OUTPUT_DIR / "candidate_article_ids.csv", index=False, encoding="utf-8-sig")

summary = pd.DataFrame([{
    "press": DATA_PATH.stem,
    "source_articles": len(source["document"]),
    "processed_articles": len(documents),
    "rule_articles_detected": rule_result.loc[rule_hit, "doc_id"].nunique(),
    "martial_law_articles_excluded": rule_result.loc[martial_law, "doc_id"].nunique(),
    "candidate_articles": len(candidate_doc_ids),
    "candidate_sentences": int(candidate_sentence.sum()),
}])
summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False, encoding="utf-8-sig")
display(summary)
# display(candidate_rule_result.tail(2))
print(f"저장 완료: {OUTPUT_DIR}")

,press,source_articles,processed_articles,rule_articles_detected,martial_law_articles_excluded,candidate_articles,candidate_sentences
0,노컷뉴스,54008,54008,33513,3007,30689,188204


저장 완료: C:\work\ner\data\filter\rule_detect_data\20261117_190820_노컷뉴스


##### press_list 

In [11]:
press_list = ["일간투데이","충청일보","강원일보","경기일보","경향신문","노컷뉴스","뉴스1","뉴스핌","뉴시스","디지털타임스","마이데일리","부산일보","서울신문","세계일보","스포츠경향","스포츠투데이","아시아투데이","이데일리","이투데이"]

RULE_OUTPUT_ROOT = Path(r"C:\work\ner\data\filter\rule_detect_data")
CHUNK_SIZE = 1000 

# 룰 검출 
refer_data = load_refer_data()
error_data = refer_data["error_data"]
high_frequency_error_data = error_data.loc[error_data["freq_quartile"].isin(["Q4", "Q3", "Q2", "Q1"])].copy()

from time import perf_counter
from tqdm.auto import tqdm

def sentence_data_from_documents(data, docs):
    source_id = data.get("id", DATA_PATH.stem)
    rows = []
    for doc in tqdm(docs, desc="문서 > 문장 변환"):
        for paragraph in doc["paragraph"]:
            for sentence in paragraph["sentence"]:
                rows.append({
                    "id": source_id,
                    "doc_id": doc["id"],
                    "sentence_id": sentence["id"],
                    "form": sentence["form"],
                })
    return pd.DataFrame(rows)

def run_required_rules(data, docs):
    started = perf_counter()
    output = sentence_data_from_documents(data, docs)
    print(f"문장 변환 완료: {perf_counter() - started:.1f}초")

    sentence_rules = (ext_s001, ext_s002, ext_s003)
    for rule in tqdm(sentence_rules, desc="문장 룰"):
        rule_started = perf_counter()
        output = rule(output)
        print(f"{rule.__name__} 완료: {perf_counter() - rule_started:.1f}초")

    started = perf_counter()
    output = entity_data(output)
    print(f"개체명 변환 완료: {perf_counter() - started:.1f}초")

    entity_rules = (
        (ext_e001, high_frequency_error_data),
        (ext_e002, None),
        (ext_e003, refer_data["mistag_data"]),
        (ext_e004, refer_data["tag_data"]),
        (ext_e005, refer_data["tag_data"]),
        (ext_e006, refer_data["tag_data"]),
    )

    for rule, rule_data in tqdm(entity_rules, desc="개체명 룰"):
        rule_started = perf_counter()
        output = rule(output, rule_data) if rule_data is not None else rule(output)
        print(f"{rule.__name__} 완료: {perf_counter() - rule_started:.1f}초")

    return output

In [ ]:
for press in press_list:
    DATA_PATH = Path(rf"C:\work\ner\data\org\all_pass_ner_tagged_5\{press}.json")
    RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S") + f"_{press}"

    OUTPUT_DIR = RULE_OUTPUT_ROOT / RUN_ID
    CHUNK_DIR = RULE_OUTPUT_ROOT / RUN_ID / DATA_PATH.stem
    OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
    CHUNK_DIR.mkdir(parents=True, exist_ok=False)

    source = json.loads(DATA_PATH.read_text(encoding="utf-8-sig"))
    documents = source["document"]
    print({"source": len(documents), "to_process": len(documents), "output": str(OUTPUT_DIR)})


    # 룰 검출 
    document_chunks = [
        documents[start:start + CHUNK_SIZE]
        for start in range(0, len(documents), CHUNK_SIZE)
    ]

    chunk_paths = []
    for chunk_number, document_chunk in enumerate(
        tqdm(document_chunks, desc="문서 청크 처리"),
        start=1,
    ):
        chunk_result = run_required_rules(source, document_chunk)
        chunk_path = CHUNK_DIR / f"{DATA_PATH.stem}_chunk_{chunk_number:04d}.pkl"
        chunk_result.to_pickle(chunk_path)
        chunk_paths.append(chunk_path)
        print(f"청크 {chunk_number}/{len(document_chunks)} 저장 완료: {chunk_path}")
        del chunk_result

    # 청크 Pickle을 병합하여 기존 최종 결과 위치에 저장
    rule_result = pd.concat(
        (pd.read_pickle(chunk_path) for chunk_path in chunk_paths),
        ignore_index=True,
    )
    final_result_path = OUTPUT_DIR / f"{DATA_PATH.stem}.pkl"
    rule_result.to_pickle(final_result_path)
    print(f"최종 병합 저장 완료: {final_result_path}")
    print(rule_result.shape, rule_result.doc_id.nunique())

    # 구조화 및 저장 
    base_columns = ["id", "doc_id", "sentence_id", "entity_id", "form", "entity_tagging", "entity_name", "entity_type"]
    rule_columns = [col for col in rule_result.columns if col.startswith(("EXT_S", "EXT_E")) and not col.endswith("_PATTERN")]
    rule_pattern_columns = [f"{col}_PATTERN" for col in rule_columns if f"{col}_PATTERN" in rule_result.columns]
    save_columns = [
        *[col for col in base_columns if col in rule_result.columns],
        *rule_columns,
        *rule_pattern_columns,
    ]
    rule_result = rule_result.loc[:, save_columns].copy()
    rule_result.to_pickle(final_result_path)

    rule_hit = rule_result[rule_columns].eq(True).any(axis=1)
    martial_law = rule_result["EXT_S001"].eq(True)

    # 룰 1개 이상 true & 오류 빈도 높았던 건 (계엄제외)
    candidate_sentence = rule_hit & rule_result["EXT_E001"].eq(True) & ~martial_law
    candidate_doc_ids = set(rule_result.loc[candidate_sentence, "doc_id"])

    # 후보 문장의 룰 결과 확인용 산출물
    candidate_columns = [*base_columns, *rule_columns, *rule_pattern_columns]
    candidate_rule_result = rule_result.loc[candidate_sentence, [col for col in candidate_columns if col in rule_result.columns]].copy()
    candidate_rule_result.to_csv(OUTPUT_DIR / "candidate_rule_results.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({"doc_id": sorted(candidate_doc_ids)}).to_csv(OUTPUT_DIR / "candidate_article_ids.csv", index=False, encoding="utf-8-sig")

    summary = pd.DataFrame([{
        "press": DATA_PATH.stem,
        "source_articles": len(source["document"]),
        "processed_articles": len(documents),
        "rule_articles_detected": rule_result.loc[rule_hit, "doc_id"].nunique(),
        "martial_law_articles_excluded": rule_result.loc[martial_law, "doc_id"].nunique(),
        "candidate_articles": len(candidate_doc_ids),
        "candidate_sentences": int(candidate_sentence.sum()),
    }])
    summary.to_csv(OUTPUT_DIR / "run_summary.csv", index=False, encoding="utf-8-sig")
    display(summary)
    # display(candidate_rule_result.tail(2))
    print(f"저장 완료: {OUTPUT_DIR}")